# **Assignment 6: Retrieval-Augmented Generation (RAG) - Machine Learning Knowledge Assistant**

In [14]:
#Install Dependencies
!pip install -q pypdf langchain tiktoken

In [15]:
!pip install -q pypdf langchain langchain-text-splitters

**Part 1 — Data Understanding & Preprocessing**

In [16]:
#Import Libraries
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = "/content/intro-to-ml.pdf"

# Load PDF & Extract Text

reader = PdfReader(pdf_path)

num_pages = len(reader.pages)
print(f"Total number of pages: {num_pages}")

Total number of pages: 392


In [17]:
#Page-wise Text Distribution
page_lengths = []

for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    page_lengths.append(len(text))

print("Min chars/page:", min(page_lengths))
print("Max chars/page:", max(page_lengths))
print("Avg chars/page:", sum(page_lengths)/len(page_lengths))

Min chars/page: 0
Max chars/page: 4604
Avg chars/page: 1765.5484693877552


In [18]:
#Detect Empty or Low-Quality Pages
empty_pages = [i for i, l in enumerate(page_lengths) if l < 50]
print(f"Empty/low-text pages: {len(empty_pages)}")
print("Page indices:", empty_pages[:10])

Empty/low-text pages: 5
Page indices: [1, 143, 223, 317, 335]


In [19]:
page_index = empty_pages[0]
print(reader.pages[page_index])

{'/Type': '/Page', '/CropBox': [0, 0, 504, 661.44], '/MediaBox': [0, 0, 504, 661.44], '/Rotate': 0, '/Resources': {'/XObject': {'/X3': IndirectObject(3, 0, 138429782477040)}}, '/Contents': IndirectObject(2, 0, 138429782477040), '/ArtBox': [0, 0, 504, 661.44], '/LastModified': "D:20160921095412-06'00'", '/Parent': IndirectObject(2687, 0, 138429782477040)}


In [20]:
# Extract text from all pages and filter the low quality pages
full_text = ""
skipped_pages = []

for i, page in enumerate(reader.pages):
    try:
        text = page.extract_text()

        # Clean basic whitespace
        if text:
            cleaned_text = text.strip()

            # Filter low-quality pages
            if len(cleaned_text) > 50:   # threshold
                full_text += cleaned_text + "\n"
            else:
                skipped_pages.append(i)
                print(f"Skipping page {i} (low text)")
        else:
            skipped_pages.append(i)
            print(f"No text found on page {i}")

    except Exception as e:
        skipped_pages.append(i)
        print(f"Error reading page {i}: {e}")

print(f"\nTotal skipped pages: {len(skipped_pages)}")

No text found on page 1
No text found on page 143
No text found on page 223
No text found on page 317
No text found on page 335

Total skipped pages: 5


In [21]:
#Character & Token Analysis
total_chars = len(full_text)
total_words = len(full_text.split())

print(f"Total characters: {total_chars}")
print(f"Total words: {total_words}")

Total characters: 692480
Total words: 103400


In [22]:
#Duplicate / Repeated Content Detection
lines = full_text.split("\n")
duplicates = len(lines) - len(set(lines))
print(f"Duplicate lines: {duplicates}")

Duplicate lines: 1256


In [23]:
from collections import Counter

lines = full_text.split("\n")
line_counts = Counter(lines)

duplicates = [line for line, count in line_counts.items() if count > 2]

print("Highly repeated lines:\n")
for line in duplicates[:50]:
    print(f"'{line}' → repeated {line_counts[line]} times")

Highly repeated lines:

'In[2]:' → repeated 7 times
'import numpy as np' → repeated 3 times
'Out[2]:' → repeated 5 times
'In[3]:' → repeated 7 times
'Out[3]:' → repeated 5 times
'In[4]:' → repeated 7 times
'Out[4]:' → repeated 4 times
'In[5]:' → repeated 7 times
'Out[5]:' → repeated 6 times
'In[6]:' → repeated 7 times
'In[7]:' → repeated 7 times
'import pandas as pd' → repeated 6 times
'In[8]:' → repeated 7 times
'In[9]:' → repeated 7 times
'Out[9]:' → repeated 4 times
'In[10]:' → repeated 7 times
'from sklearn.datasets import load_iris' → repeated 4 times
'In[11]:' → repeated 7 times
'Out[11]:' → repeated 4 times
'In[12]:' → repeated 7 times
'Out[12]:' → repeated 6 times
'In[13]:' → repeated 7 times
'Out[13]:' → repeated 5 times
'In[14]:' → repeated 7 times
'Out[14]:' → repeated 4 times
'In[15]:' → repeated 7 times
'Out[15]:' → repeated 5 times
'In[16]:' → repeated 7 times
'Out[16]:' → repeated 4 times
'In[17]:' → repeated 7 times
'Out[17]:' → repeated 4 times
'In[18]:' → repeated 7 t

In [24]:
#Detect Formatting Noise Patterns
import re
from collections import Counter

lines = full_text.split("\n")

noise_patterns = {
    "jupyter_input": r'^In\[\d+\]:',
    "jupyter_output": r'^Out\[\d+\]:',
    "page_numbers": r'^\d+$',
    "short_noise": r'^[^a-zA-Z0-9]{1,5}$'
}

noise_counts = Counter()

for line in lines:
    stripped = line.strip()
    for key, pattern in noise_patterns.items():
        if re.match(pattern, stripped):
            noise_counts[key] += 1

print("Detected Formatting Noise:\n")
for k, v in noise_counts.items():
    print(f"{k}: {v}")

Detected Formatting Noise:

page_numbers: 10
jupyter_input: 453
jupyter_output: 245
short_noise: 7


In [25]:
#Inspect Suspicious Lines
suspicious_lines = []

for line in lines:
    stripped = line.strip()

    if (
        re.match(r'^In\[\d+\]:', stripped) or
        re.match(r'^Out\[\d+\]:', stripped) or
        len(stripped) < 5
    ):
        suspicious_lines.append(stripped)

print("Sample suspicious lines:\n")
for l in suspicious_lines[:20]:
    print(l)

Sample suspicious lines:

iii
vii
1
In[2]:
Out[2]:
x:
In[3]:
Out[3]:
In[4]:
Out[4]:
In[5]:
Out[5]:
In[6]:
In[7]:
}
In[8]:
In[9]:
Out[9]:
In[10]:
In[11]:


CLeaning the lines to remove noise

In [26]:
#Clean While Keeping Meaningful Code
clean_lines = []

for line in lines:
    stripped = line.strip()

    # Remove formatting noise
    if re.match(r'^In\[\d+\]:', stripped):
        continue
    if re.match(r'^Out\[\d+\]:', stripped):
        continue
    if re.match(r'^\d+$', stripped):  # page numbers
        continue

    # Remove very short meaningless lines
    if len(stripped) < 3:
        continue

    # Keep meaningful content (including code)
    clean_lines.append(line)

clean_text = "\n".join(clean_lines)

print("Original length:", len(full_text))
print("Cleaned length:", len(clean_text))

Original length: 692480
Cleaned length: 686611


In [27]:
#Special Character / Encoding Issues
import string

weird_chars = {c for c in clean_text if c not in string.printable}

print(f"Distinct non-standard characters count: {len(weird_chars)}")
print(weird_chars)

Distinct non-standard characters count: 21
{'│', '‐', '”', 'ü', '—', '├', '×', '└', '·', 'ê', '“', 'ɣ', '–', 'ŷ', '─', '®', '’', 'ǁ', '•', '©', '…'}


In [28]:
#Special characters count
from collections import Counter

char_counts = Counter(c for c in clean_text if c not in string.printable)

print("Most common non-standard characters:\n")
for char, count in char_counts.most_common(10):
    print(f"'{char}' → {count}")

Most common non-standard characters:

'‐' → 736
'’' → 366
'“' → 248
'”' → 248
'—' → 102
'–' → 40
'…' → 28
'•' → 27
'ŷ' → 14
'─' → 12


In [29]:
#Removing the unwanted characters
import re

def clean_special_chars(text):
    replacements = {
        "“": '"', "”": '"',
        "’": "'",
        "–": "-", "—": "-", "‐": "-",
        "…": "...",
        "×": "x",
        "·": ".",
    }

    # Replace known useful characters
    for k, v in replacements.items():
        text = text.replace(k, v)

    # Remove unwanted symbols
    text = re.sub(r'[•─└├│©®]', '', text)

    # Remove remaining non-ascii (encoding noise)
    text = text.encode("ascii", "ignore").decode()

    return text

clean_text = clean_special_chars(clean_text)

print("Cleaned text preview:\n", clean_text[:500])
print("Cleaned length:", len(clean_text))

Cleaned text preview:
 Andreas C. Mller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS
Andreas C. Mller and Sarah Guido
Introduction to Machine Learning
with Python
A Guide for Data Scientists
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing
978-1-449-36941-5
[LSI]
Introduction to Machine Learning with Python
by Andreas C. Mller and Sarah Guido
Copyright  2017 Sarah Guido, Andreas Mller. All rights reserved.
Printed in the United States o
Cleaned length: 686587


In [30]:
#Sentence Length Distribution
sentences = clean_text.split(".")
lengths = [len(s.split()) for s in sentences]

print("Avg sentence length:", sum(lengths)/len(lengths))

Avg sentence length: 13.019315944881889


In [31]:
#Keyword / Topic Frequency
from collections import Counter

words = clean_text.lower().split()
common_words = Counter(words).most_common(20)

print("Top frequent words:")
print(common_words)

Top frequent words:
[('the', 5931), ('of', 2492), ('a', 2127), ('to', 2110), ('and', 2101), ('in', 1650), ('is', 1574), ('we', 1180), ('for', 1085), ('that', 996), ('are', 794), ('this', 743), ('data', 741), ('as', 738), ('=', 721), ('can', 633), ('on', 621), ('with', 579), ('0', 575), ('.', 542)]


In [32]:
#Split Text into Chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_text(clean_text)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 784


In [33]:
#Chunk Quality Preview
for i in range(30):
    print(f"\n--- Chunk {i} ---")
    print(chunks[i][:100])


--- Chunk 0 ---
Andreas C. Mller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR 

--- Chunk 1 ---
Editor: Dawn Schanafelt
Production Editor: Kristen Brown
Copyeditor: Rachel Head
Proofreader: Jasmin

--- Chunk 2 ---
for errors or omissions, including without limitation responsibility for damages resulting from the 

--- Chunk 3 ---
Problems Machine Learning Can Solve                                                                 

--- Chunk 4 ---
SciPy                                                                                               

--- Chunk 5 ---
A First Application: Classifying Iris Species                                                       

--- Chunk 6 ---
iii
2. Supervised Learning. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 

--- Chunk 7 ---
Naive Bayes Classifiers                                                                             

--- Chunk 8 ---
Uncertainty in Multiclass Classificatio

In [34]:
# Removing first few chunks after inspection
chunks_final = chunks[33:]

print(f"Chunks after cleanup: {len(chunks_final)}")

Chunks after cleanup: 751


In [35]:
#Chunk Quality Preview
for i in range(3):
    print(f"\n--- Chunk {i} ---")
    print(chunks_final[i][:100])


--- Chunk 0 ---
CHAPTER 1
Introduction
Machine learning is about extracting knowledge from data. It is a research fi

--- Chunk 1 ---
ence on the way data-driven research is done today. The tools introduced in this book
have been appl

--- Chunk 2 ---
whose job is to move the appropriate incoming email messages to a spam folder. Y ou
could make up a 


**Part 2 — Embedding & Vector Database**

In [36]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 30.2 MB/s eta 0:00:00


In [37]:
#import libraries
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [38]:
#Load embedding model
model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Embedding Model Used**

multi-qa-MiniLM-L6-cos-v1

Reason:

Optimized for question answering retrieval

Improves relevance of extracted textbook chunks

Reduces hallucination in LLM generation stage


In [39]:
#Generate embeddings
embeddings = model.encode(
    chunks_final,
    show_progress_bar=True,
    convert_to_numpy=True
)

# FAISS requires float32
embeddings = np.array(embeddings).astype("float32")

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Embedding shape: (751, 384)


In [40]:
#Create FAISS vector database
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Total vectors stored in FAISS:", index.ntotal)

Total vectors stored in FAISS: 751


**Vector storage approach**

Embeddings stored using FAISS IndexFlatL2

Each chunk represented as a dense vector

Query is matched using L2 distance similarity

**Part 3 — Retrieval Pipeline**

In [41]:
#Retrieval function (top-k chunks)
def retrieve_chunks(query, k=5):
    # Step 1: embed query
    query_embedding = model.encode([query]).astype("float32")

    # Step 2: similarity search
    distances, indices = index.search(query_embedding, k)

    # Step 3: collect results
    results = []
    for idx in indices[0]:
        results.append(chunks_final[idx])

    return results

In [42]:
query = "What is backpropagation in neural networks?"

In [43]:
results_k3 = retrieve_chunks(query, k=3)

for i, r in enumerate(results_k3):
    print(f"\n--- Chunk {i+1} (k=3) ---\n")
    print(r[:800])


--- Chunk 1 (k=3) ---

also known as (vanilla) feed-forward neural networks, or sometimes just neural
networks.
The neural network model
MLPs can be viewed as generalizations of linear models that perform multiple stages
of processing to come to a decision.
104 | Chapter 2: Supervised Learning
Remember that the prediction by a linear regressor is given as:
 = w[0] * x[0] + w[1] * x[1] + ... + w[p] * x[p] + b
In plain English,  is a weighted sum of the input features x[0] to x[p], weighted by
the learned coefficients w[0] to w[p]. We could visualize this graphically as shown in
Figure 2-44:
display(mglearn.plots.plot_logistic_regression_graph())
Figure 2-44. Visualization of logistic regression, where input features and predictions are
shown as nodes, and the coefficients are connections between the nodes
Here, e

--- Chunk 2 (k=3) ---

kernel has only one parameter, gamma, which is the inverse of the width of the Gaus-
sian kernel. gamma and C both control the complexity of the model,

In [44]:
results_k5 = retrieve_chunks(query, k=5)

for i, r in enumerate(results_k5):
    print(f"\n--- Chunk {i+1} (k=5) ---\n")
    print(r[:800])


--- Chunk 1 (k=5) ---

also known as (vanilla) feed-forward neural networks, or sometimes just neural
networks.
The neural network model
MLPs can be viewed as generalizations of linear models that perform multiple stages
of processing to come to a decision.
104 | Chapter 2: Supervised Learning
Remember that the prediction by a linear regressor is given as:
 = w[0] * x[0] + w[1] * x[1] + ... + w[p] * x[p] + b
In plain English,  is a weighted sum of the input features x[0] to x[p], weighted by
the learned coefficients w[0] to w[p]. We could visualize this graphically as shown in
Figure 2-44:
display(mglearn.plots.plot_logistic_regression_graph())
Figure 2-44. Visualization of logistic regression, where input features and predictions are
shown as nodes, and the coefficients are connections between the nodes
Here, e

--- Chunk 2 (k=5) ---

kernel has only one parameter, gamma, which is the inverse of the width of the Gaus-
sian kernel. gamma and C both control the complexity of the model,

In [45]:
results_k7 = retrieve_chunks(query, k=7)

for i, r in enumerate(results_k7):
    print(f"\n--- Chunk {i+1} (k=7) ---\n")
    print(r[:800])


--- Chunk 1 (k=7) ---

also known as (vanilla) feed-forward neural networks, or sometimes just neural
networks.
The neural network model
MLPs can be viewed as generalizations of linear models that perform multiple stages
of processing to come to a decision.
104 | Chapter 2: Supervised Learning
Remember that the prediction by a linear regressor is given as:
 = w[0] * x[0] + w[1] * x[1] + ... + w[p] * x[p] + b
In plain English,  is a weighted sum of the input features x[0] to x[p], weighted by
the learned coefficients w[0] to w[p]. We could visualize this graphically as shown in
Figure 2-44:
display(mglearn.plots.plot_logistic_regression_graph())
Figure 2-44. Visualization of logistic regression, where input features and predictions are
shown as nodes, and the coefficients are connections between the nodes
Here, e

--- Chunk 2 (k=7) ---

kernel has only one parameter, gamma, which is the inverse of the width of the Gaus-
sian kernel. gamma and C both control the complexity of the model,

In [46]:
query = "What is random forest and how does it work?"

results = retrieve_chunks(query, k=5)

for i, chunk in enumerate(results):
    print(f"\n--- Chunk {i+1} ---\n")
    print(chunk[:800])


--- Chunk 1 ---

Breast Cancer dataset
As you can see, the random forest gives nonzero importance to many more features
than the single tree. Similarly to the single decision tree, the random forest also gives
a lot of importance to the "worst radius" feature, but it actually chooses "worst perim-
eter" to be the most informative feature overall. The randomness in building the ran-
dom forest forces the algorithm to consider many possible explanations, the result
being that the random forest captures a much broader picture of the data than a sin-
gle tree.
Strengths, weaknesses, and parameters.    Random forests for regression and classifica-
tion are currently among the most widely used machine learning methods. They are
very powerful, often work well without heavy tuning of the parameters, and don't
requ

--- Chunk 2 ---

an acceptable job of predicting the target, and should also be different from the other
trees. Random forests get their name from injecting randomness into the tre

**Part 4 — Answer Generation (RAG)**

In [47]:
!pip install -q google-generativeai

In [60]:
import google.generativeai as genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

genai.configure(api_key=api_key)

llm = genai.GenerativeModel("gemini-flash-latest")

In [61]:
#Retrieve + Build Context
def build_context(query, k=5):
    retrieved_chunks = retrieve_chunks(query, k=k)

    context = "\n\n".join(retrieved_chunks)

    return context

In [62]:
#Prompt Design
def build_prompt(context, question):
    prompt = f"""
You are an expert Machine Learning tutor.

You must answer ONLY using the provided context below.
If the answer is not in the context, say:
"I don't know based on the provided document."

RULES:
- Do NOT use external knowledge
- Be accurate and concise
- Do NOT hallucinate
- Base answer only on context

CONTEXT:
{context}

QUESTION:
{question}

FINAL ANSWER:
"""
    return prompt

In [63]:
#Generate Answer using LLM
def generate_answer(query, k=5):
    # Step 1: Retrieve context
    context = build_context(query, k)

    # Step 2: Create prompt
    prompt = build_prompt(context, query)

    # Step 3: Get LLM response
    response = llm.generate_content(prompt)

    return response.text

In [64]:
#Test the RAG system
query = "What is a backpropagation?"

answer = generate_answer(query, k=5)

print("QUESTION:\n", query)
print("\nANSWER:\n", answer)

QUESTION:
 What is a backpropagation?

ANSWER:
 I don't know based on the provided document.


In [65]:
#Test multiple queries
queries = [
    "What is a neural network?",
    "How does random forest work?",
    "What is gradient boosting?",
]

for q in queries:
    print("\n" + "="*80)
    print("Q:", q)
    print("\nA:", generate_answer(q, k=5))


Q: What is a neural network?

A: Neural networks are a family of algorithms also known as "deep learning." Multilayer perceptrons (MLPs) are referred to as (vanilla) feed-forward neural networks, or sometimes just neural networks. They can be viewed as generalizations of linear models that perform multiple stages of processing to come to a decision.

Q: How does random forest work?

A: Random forests work by building a collection of decision trees that are constructed independently from each other. To ensure that each tree is distinct and captures a broad picture of the data, the algorithm injects randomness into the building process in two ways:

1.  **Data Point Selection:** For each tree, the algorithm takes a "bootstrap sample" of the data to determine which points are used for training.
2.  **Feature Selection:** The algorithm selects specific features to use in each split test.

By making these different random choices, the forest considers many possible explanations for the dat

**Part 5 — End-to-End RAG Application**

In [66]:
#REG Function
def rag_pipeline(query, k=5):

    # Step 1: Retrieve relevant chunks
    retrieved_chunks = retrieve_chunks(query, k=k)

    # Step 2: Build context
    context = "\n\n".join(retrieved_chunks)

    # Step 3: Create grounded prompt
    prompt = f"""
You are a Machine Learning expert assistant.

Answer ONLY using the provided context.
If the answer is not in the context, say:
"I don't know based on the given document."

Rules:
- Do NOT use outside knowledge
- Be concise and accurate
- Base answer strictly on context

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""

    # Step 4: Generate response from LLM
    response = llm.generate_content(prompt)

    return response.text

In [67]:
#Real-time Question Answering Function
def ask_question():
    print("RAG Machine Learning Assistant (type 'exit' to stop)\n")

    while True:
        query = input("Ask a question: ")

        if query.lower() == "exit":
            print("Exiting RAG system...")
            break

        answer = rag_pipeline(query, k=5)

        print("\n Answer:\n", answer)
        print("\n" + "-"*80 + "\n")

In [68]:
#Run the system (Interactive mode)

ask_question()

RAG Machine Learning Assistant (type 'exit' to stop)

Ask a question: What is gradient boosting?

 Answer:
 Gradient boosting is a method that combines many simple models, known as weak learners (such as shallow trees with a depth of one to five), to iteratively improve performance. Each tree provides good predictions on only part of the data, and more trees are added to correct the mistakes of previous trees. Unlike random forests, there is no randomization; instead, strong pre-pruning is used.

--------------------------------------------------------------------------------

Ask a question: what is deeep learning ?

 Answer:
 I don't know based on the given document.

--------------------------------------------------------------------------------

Ask a question: what is machine learning ?

 Answer:
 Machine learning is about extracting knowledge from data. It is a research field at the intersection of statistics, artificial intelligence, and computer science, and is also known as p

In [69]:
test_queries = [
    "What is a neural network?",
    "Explain random forest algorithm",
    "What is gradient boosting?",
    "How does backpropagation work?",
]

for q in test_queries:
    print("\n" + "="*80)
    print("QUESTION:", q)
    print("\nANSWER:\n", rag_pipeline(q, k=5))


QUESTION: What is a neural network?

ANSWER:
 A neural network is a family of algorithms, also known as "deep learning" or multilayer perceptrons (MLPs), which can be viewed as generalizations of linear models that perform multiple stages of processing to come to a decision. Multilayer perceptrons are also referred to as (vanilla) feed-forward neural networks.

QUESTION: Explain random forest algorithm

ANSWER:
 Based on the provided context, the random forest algorithm is explained as follows:

*   **Definition and Goal:** A random forest is a collection of decision trees that are built independently. The goal is for each tree to do an acceptable job of predicting the target while remaining distinct from the other trees.
*   **Injecting Randomness:** To ensure trees are different, randomness is injected in two ways:
    1.  **Selecting data points:** Each tree is built using a "bootstrap sample" of the data.
    2.  **Selecting features:** Random choices are made when selecting the f